# 🔬 TALOS V1.5 — FASE 1A.2: PILOTO EXPERIMENTAL COLAB-FIRST

### Ingesta, Extracción, Chunking y Vectorización Factorial (Joyanes 65 Páginas)

**Gobernanza y Aislamiento:**
- `PDF_PROCESSING_LOCAL = FORBIDDEN` (La PC local no procesa PDFs)
- `PDF_PROCESSING_COLAB = REQUIRED` (Google Colab con GPU T4 asume el cómputo pesado)
- `PRODUCTION_DB_MODIFIED = FALSE` (No altera bases de producción)
- `CANDIDATE_SELECTION = NOT_AUTHORIZED` (Este piloto valida la infraestructura experimental)
- `FAIL-CLOSED` ante cualquier discrepancia criptográfica de hash SHA-256.

**Cortes Autorizados de Joyanes (`1217dd5e...`):**
- `J-01`: Páginas 11 a 18 (Preliminares / Índice general / Outlier PROB-1A-03 de 47k chars)
- `J-02`: Páginas 35 a 45 (Concepto de algoritmo, diseño descendente, listas y encabezados)
- `J-03`: Páginas 135 a 150 (Estructuras de control condicionales simples y dobles)
- `J-04`: Páginas 485 a 505 (Punteros, estructuras struct y gestión dinámica en C)
- `J-05`: Páginas 540 a 548 (Complejidad algorítmica y notación asintótica Big-O)

In [ ]:
# =============================================================================
# Celda 1: Instalación de Dependencias Requeridas
# =============================================================================
!pip install -q pypdf pymupdf sentence-transformers accelerate tabulate pandas
print("✓ Dependencias instaladas con éxito.")

In [ ]:
# =============================================================================
# Celda 2: Diagnóstico de Entorno y GPU
# =============================================================================
import os, sys, time, json, hashlib, gc, re, shutil
from pathlib import Path
import torch
import numpy as np
import pandas as pd

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
cuda_ok = torch.cuda.is_available()
print("CUDA Available:", cuda_ok)
if cuda_ok:
    gpu_name = torch.cuda.get_device_name(0)
    vram_mb = torch.cuda.get_device_properties(0).total_memory // (1024 * 1024)
    print(f"GPU: {gpu_name} ({vram_mb} MB VRAM)")
    DEVICE = "cuda"
else:
    print("ADVERTENCIA: GPU no detectada. Operando en CPU.")
    DEVICE = "cpu"

In [ ]:
# =============================================================================
# Celda 3: Montaje de Google Drive y Auto-Localización del Fixture Joyanes
# =============================================================================
try:
    from google.colab import drive
    if not os.path.exists("/content/drive/MyDrive"):
        drive.mount("/content/drive")
        print("✓ Google Drive montado.")
except Exception as e:
    print(f"ℹ Google Drive no disponible: {e}")

EXPECTED_JOYANES_SHA256 = "1217dd5e6038f150f454198b9645b7f4b89929ff88ba3744ca2c446082f4fb47"
EXPECTED_JOYANES_SIZE = 21897183
EXPECTED_PAGES = 802

def compute_sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, "rb") as f:
        while chunk := f.read(65536):
            h.update(chunk)
    return h.hexdigest()

def auto_locate_joyanes_pdf():
    candidates = [
        Path("corpus_inputs/Fundamentos_de_programación_4ta_Edición_Luis_Joyanes_Aguilar_2.pdf"),
        Path("Fundamentos_de_programación_4ta_Edición_Luis_Joyanes_Aguilar_2.pdf"),
        Path("/content/Fundamentos_de_programación_4ta_Edición_Luis_Joyanes_Aguilar_2.pdf"),
        Path("/content/corpus_inputs/Fundamentos_de_programación_4ta_Edición_Luis_Joyanes_Aguilar_2.pdf"),
    ]
    for c in candidates:
        if c.exists() and c.is_file() and c.stat().st_size > 1000:
            print(f"[*] Probando candidato directo: {c}")
            if compute_sha256(str(c)) == EXPECTED_JOYANES_SHA256:
                return str(c)
    
    # Búsqueda en Google Drive si está montado
    drive_root = Path("/content/drive/MyDrive")
    if drive_root.exists():
        print("[*] Buscando PDF de Joyanes en Google Drive...")
        for match in drive_root.glob("**/*Joyanes*.pdf"):
            try:
                if match.is_file() and match.stat().st_size == EXPECTED_JOYANES_SIZE:
                    print(f"[*] Verificando hash SHA-256 de {match}...")
                    if compute_sha256(str(match)) == EXPECTED_JOYANES_SHA256:
                        print(f"✓ Encontrado y verificado en Google Drive: {match}")
                        return str(match)
            except Exception:
                pass
    
    # Búsqueda en /content
    content_root = Path("/content")
    for match in content_root.glob("**/*Joyanes*.pdf"):
        try:
            if match.is_file() and compute_sha256(str(match)) == EXPECTED_JOYANES_SHA256:
                return str(match)
        except Exception:
            pass
    return None

PDF_PATH = auto_locate_joyanes_pdf()
if PDF_PATH:
    print(f"\n[+] FIXTURE VERIFICADO CON ÉXITO: {PDF_PATH}")
    print(f"    SHA-256: {EXPECTED_JOYANES_SHA256} (MATCH EXACTO)")
else:
    print("\n[!] ERROR CRÍTICO: No se encontró el PDF verificado de Joyanes Aguilar.")
    print("    Por favor, sube 'Fundamentos_de_programación_4ta_Edición_Luis_Joyanes_Aguilar_2.pdf' a tu Drive o a /content/")
    raise FileNotFoundError("INPUT_FIXTURE_NOT_FOUND")

In [ ]:
# =============================================================================
# Celda 4: Inicialización de Configuraciones y Manifiestos Canónicos
# =============================================================================
INPUT_MANIFEST = {
  "$schema": "https://raw.githubusercontent.com/guarnieriagustincyber/TALOS/main/experiments/v1_5_phase1a2/schemas/input_manifest.schema.json",
  "batch_id": "batch_1a2_exp_corpus_v1",
  "created_at": "2026-10-05T02:00:00Z",
  "governance": {
    "status": "APPROVED_FOR_1A2_COLAB",
    "pdf_processing_local": "FORBIDDEN FOR THIS EXPERIMENTAL CAMPAIGN",
    "pdf_processing_colab": "REQUIRED",
    "pilot_scope": "doc_joyanes_fundamentos_4ed_only"
  },
  "documents": [
    {
      "document_id": "doc_joyanes_fundamentos_4ed",
      "source_id": "src_joyanes_fundamentos",
      "filename": "Fundamentos_de_programación_4ta_Edición_Luis_Joyanes_Aguilar_2.pdf",
      "edition_version": "4ta Edición (McGraw-Hill)",
      "language": "es",
      "input_fixture_status": "VERIFIED",
      "local_path": "data/sources/general/Fundamentos_de_programación_4ta_Edición_Luis_Joyanes_Aguilar_2.pdf",
      "page_count_expected": 802,
      "sha256": "1217dd5e6038f150f454198b9645b7f4b89929ff88ba3744ca2c446082f4fb47",
      "target_page_slices": [
        {
          "slice_id": "J-01",
          "pages": [
            11,
            18
          ],
          "rationale": "Preliminares e Índice General (TOC outlier PROB-1A-03, 47k chars)"
        },
        {
          "slice_id": "J-02",
          "pages": [
            35,
            45
          ],
          "rationale": "Prosa, listas y jerarquía básica de encabezados H1-H3"
        },
        {
          "slice_id": "J-03",
          "pages": [
            135,
            150
          ],
          "rationale": "Estructuras de control y pseudocódigo estructurado"
        },
        {
          "slice_id": "J-04",
          "pages": [
            485,
            505
          ],
          "rationale": "Punteros, estructuras y bloques densos de código C"
        },
        {
          "slice_id": "J-05",
          "pages": [
            540,
            548
          ],
          "rationale": "Complejidad algorítmica y notación matemática Big-O"
        }
      ]
    },
    {
      "document_id": "doc_tanenbaum_sistemas_operativos_3ed",
      "source_id": "src_tanenbaum_sistemas_operativos",
      "filename": "Sistemas_Operativos_Modernos_3ra_Edicion_Tanenbaum.pdf",
      "edition_version": "3ra Edición (Pearson)",
      "language": "es",
      "input_fixture_status": "MISSING",
      "staging_required_before_full_corpus": true,
      "page_count_expected": 1104,
      "sha256": "82f808db394ad24346768393d9894e63717df3d26780c85c2c4314c1d636b042",
      "target_page_slices": [
        {
          "slice_id": "T-01",
          "pages": [
            40,
            55
          ],
          "rationale": "Arquitectura de sistemas y llamadas al sistema POSIX"
        },
        {
          "slice_id": "T-02",
          "pages": [
            85,
            105
          ],
          "rationale": "Concurrencia, sincronización de procesos y tablas de estados"
        },
        {
          "slice_id": "T-03",
          "pages": [
            190,
            207
          ],
          "rationale": "Gestión de memoria, tablas de páginas multinivel y reemplazo"
        }
      ]
    }
  ]
}

PIPELINE_CONFIGS = {
  "$schema": "talos-pipeline-configs-schema-v1",
  "version": "1.1.0",
  "governance": {
    "status": "INITIAL_EXPERIMENT_MATRIX",
    "pdf_processing_local": "FORBIDDEN FOR THIS EXPERIMENTAL CAMPAIGN",
    "pdf_processing_colab": "REQUIRED"
  },
  "controls": {
    "retrieval_control_v1": "FTS5_BM25",
    "document_control_v1": {
      "extractor": "pypdf",
      "chunker": "fixed_character_sliding_500c"
    },
    "embedding_control_v1": {
      "model_id": "Qwen/Qwen3-Embedding-0.6B",
      "dimensions": 1024,
      "pooling": "lasttoken"
    }
  },
  "pipelines": [
    {
      "pipeline_id": "EXP-EXT-01",
      "classification": "FACTORIAL_CONTROL",
      "variable_isolated": "extractor",
      "description": "Control histórico TALOS V1 (pypdf + chunking léxico 500c + Qwen3 1024d)",
      "is_control": true,
      "extractor": {
        "engine": "pypdf",
        "version": "5.x"
      },
      "chunking": {
        "strategy": "fixed_character_sliding",
        "nominal_chars": 500,
        "overlap_chars": 50
      },
      "embedding": {
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "dimensions": 1024,
        "pooling": "lasttoken",
        "normalize_l2": true,
        "max_sequence_tokens": 512,
        "inference_dtype": "float32",
        "storage_dtype": "float32"
      }
    },
    {
      "pipeline_id": "EXP-EXT-02",
      "classification": "FACTORIAL_ISOLATION",
      "variable_isolated": "extractor",
      "description": "PyMuPDF con chunking léxico C1 y Qwen3 M1 (aislamiento de motor de extracción C rápido)",
      "is_control": false,
      "extractor": {
        "engine": "pymupdf",
        "version": "1.24+"
      },
      "chunking": {
        "strategy": "fixed_character_sliding",
        "nominal_chars": 500,
        "overlap_chars": 50
      },
      "embedding": {
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "dimensions": 1024,
        "pooling": "lasttoken",
        "normalize_l2": true,
        "max_sequence_tokens": 512,
        "inference_dtype": "float32",
        "storage_dtype": "float32"
      }
    },
    {
      "pipeline_id": "EXP-EXT-03",
      "classification": "FACTORIAL_ISOLATION",
      "variable_isolated": "extractor",
      "description": "Docling con chunking léxico C1 y Qwen3 M1 (aislamiento de extracción de maquetación profunda)",
      "is_control": false,
      "extractor": {
        "engine": "docling",
        "version": "2.x",
        "table_mode": "accurate"
      },
      "chunking": {
        "strategy": "fixed_character_sliding",
        "nominal_chars": 500,
        "overlap_chars": 50
      },
      "embedding": {
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "dimensions": 1024,
        "pooling": "lasttoken",
        "normalize_l2": true,
        "max_sequence_tokens": 512,
        "inference_dtype": "float32",
        "storage_dtype": "float32"
      }
    },
    {
      "pipeline_id": "EXP-CHK-01",
      "classification": "FACTORIAL_CONTROL",
      "variable_isolated": "chunking",
      "description": "Docling con chunking léxico C1 y Qwen3 M1 (base para variación de chunking)",
      "is_control": false,
      "extractor": {
        "engine": "docling",
        "version": "2.x",
        "table_mode": "accurate"
      },
      "chunking": {
        "strategy": "fixed_character_sliding",
        "nominal_chars": 500,
        "overlap_chars": 50
      },
      "embedding": {
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "dimensions": 1024,
        "pooling": "lasttoken",
        "normalize_l2": true,
        "max_sequence_tokens": 512,
        "inference_dtype": "float32",
        "storage_dtype": "float32"
      }
    },
    {
      "pipeline_id": "EXP-CHK-02",
      "classification": "FACTORIAL_ISOLATION",
      "variable_isolated": "chunking",
      "description": "Docling con chunking por secciones y encabezados (Section-Aware / Heading-Path)",
      "is_control": false,
      "extractor": {
        "engine": "docling",
        "version": "2.x",
        "table_mode": "accurate"
      },
      "chunking": {
        "strategy": "section_aware_heading_path",
        "max_chars_ceiling": 1500,
        "prepend_heading_path": true
      },
      "embedding": {
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "dimensions": 1024,
        "pooling": "lasttoken",
        "normalize_l2": true,
        "max_sequence_tokens": 512,
        "inference_dtype": "float32",
        "storage_dtype": "float32"
      }
    },
    {
      "pipeline_id": "EXP-CHK-03",
      "classification": "FACTORIAL_ISOLATION",
      "variable_isolated": "chunking",
      "description": "Docling con preservación atómica de párrafos y código (C3)",
      "is_control": false,
      "extractor": {
        "engine": "docling",
        "version": "2.x",
        "table_mode": "accurate"
      },
      "chunking": {
        "strategy": "paragraph_code_atomic",
        "max_chars_ceiling": 2000,
        "atomic_tables": true,
        "atomic_code": true
      },
      "embedding": {
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "dimensions": 1024,
        "pooling": "lasttoken",
        "normalize_l2": true,
        "max_sequence_tokens": 512,
        "inference_dtype": "float32",
        "storage_dtype": "float32"
      }
    },
    {
      "pipeline_id": "EXP-EMB-01",
      "classification": "FACTORIAL_CONTROL",
      "variable_isolated": "embedding",
      "description": "PyMuPDF + C2 con Qwen3 M1 (base para variación de embeddings)",
      "is_control": false,
      "extractor": {
        "engine": "pymupdf",
        "version": "1.24+"
      },
      "chunking": {
        "strategy": "section_aware_heading_path",
        "max_chars_ceiling": 1500,
        "prepend_heading_path": true
      },
      "embedding": {
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "dimensions": 1024,
        "pooling": "lasttoken",
        "normalize_l2": true,
        "max_sequence_tokens": 512,
        "inference_dtype": "float32",
        "storage_dtype": "float32"
      }
    },
    {
      "pipeline_id": "EXP-EMB-02",
      "classification": "FACTORIAL_ISOLATION",
      "variable_isolated": "embedding",
      "description": "PyMuPDF + C2 con E5-small M3 (evaluación de viabilidad de runtime local 384d)",
      "is_control": false,
      "extractor": {
        "engine": "pymupdf",
        "version": "1.24+"
      },
      "chunking": {
        "strategy": "section_aware_heading_path",
        "max_chars_ceiling": 1500,
        "prepend_heading_path": true
      },
      "embedding": {
        "model_id": "intfloat/multilingual-e5-small",
        "dimensions": 384,
        "pooling": "mean",
        "normalize_l2": true,
        "max_sequence_tokens": 512,
        "inference_dtype": "float32",
        "storage_dtype": "float32",
        "prefixes": {
          "query": "query: ",
          "passage": "passage: "
        }
      }
    },
    {
      "pipeline_id": "EXP-EMB-03",
      "classification": "FACTORIAL_ISOLATION",
      "variable_isolated": "embedding",
      "description": "PyMuPDF + C2 con BGE-M3 M2 (evaluación de contexto largo 8192 tokens)",
      "is_control": false,
      "extractor": {
        "engine": "pymupdf",
        "version": "1.24+"
      },
      "chunking": {
        "strategy": "section_aware_heading_path",
        "max_chars_ceiling": 1500,
        "prepend_heading_path": true
      },
      "embedding": {
        "model_id": "BAAI/bge-m3",
        "dimensions": 1024,
        "pooling": "cls",
        "normalize_l2": true,
        "max_sequence_tokens": 8192,
        "inference_dtype": "float32",
        "storage_dtype": "float32"
      }
    },
    {
      "pipeline_id": "PIPE-INT-01",
      "classification": "INTEGRATED_CONFIGURATION",
      "variable_isolated": "none_multivariate",
      "description": "Docling + C2 Section-Aware + BGE-M3 (Sinergia multimodal y contexto largo)",
      "is_control": false,
      "extractor": {
        "engine": "docling",
        "version": "2.x",
        "table_mode": "accurate"
      },
      "chunking": {
        "strategy": "section_aware_heading_path",
        "max_chars_ceiling": 3000,
        "prepend_heading_path": true
      },
      "embedding": {
        "model_id": "BAAI/bge-m3",
        "dimensions": 1024,
        "pooling": "cls",
        "normalize_l2": true,
        "max_sequence_tokens": 8192,
        "inference_dtype": "float32",
        "storage_dtype": "float32"
      }
    },
    {
      "pipeline_id": "PIPE-INT-03",
      "classification": "CHUNKING_AND_CONTEXT_ASSEMBLY",
      "variable_isolated": "hierarchical_architecture",
      "description": "PyMuPDF con chunking jerárquico Padre-Hijo (C4)",
      "is_control": false,
      "extractor": {
        "engine": "pymupdf",
        "version": "1.24+"
      },
      "chunking": {
        "strategy": "hierarchical_parent_child",
        "child_nominal_chars": 400,
        "parent_window_chars": 2000
      },
      "embedding": {
        "model_id": "Qwen/Qwen3-Embedding-0.6B",
        "dimensions": 1024,
        "pooling": "lasttoken",
        "normalize_l2": true,
        "max_sequence_tokens": 512,
        "inference_dtype": "float32",
        "storage_dtype": "float32"
      }
    }
  ]
}

BENCHMARK_QUERIES = [
  {
    "query_id": "Q1A2-01",
    "archetype": "ARQ-01",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-01",
    "query_text": "¿Cuál es la estructura general de contenidos y preliminares presentada en el índice de Fundamentos de Programación?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      11,
      14
    ],
    "target_section": "Contenido General / Preliminares",
    "gold_chunk_criteria": {
      "required_terms": [
        "contenido",
        "capítulo",
        "fundamentos",
        "programación"
      ],
      "structural_element_required": "text | heading",
      "minimum_char_overlap": 150
    },
    "adjudication_rubric": {
      "relevant_conditions": "Recupera la descripción o listado de capítulos preliminares de la obra.",
      "sufficient_conditions": "Permite contrastar el esquema temático de la obra."
    }
  },
  {
    "query_id": "Q1A2-02",
    "archetype": "ARQ-08",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-01",
    "query_text": "¿En qué página y capítulo del índice general se ubican las estructuras de datos y punteros?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      14,
      18
    ],
    "target_section": "Índice de Contenido / Estructuras de Datos",
    "gold_chunk_criteria": {
      "required_terms": [
        "estructuras de datos",
        "punteros",
        "capítulo"
      ],
      "structural_element_required": "table | text",
      "minimum_char_overlap": 150
    },
    "adjudication_rubric": {
      "relevant_conditions": "Identifica los números de página o capítulos correspondientes a estructuras de datos o punteros.",
      "sufficient_conditions": "Proporciona la ubicación precisa en el índice general."
    }
  },
  {
    "query_id": "Q1A2-03",
    "archetype": "ARQ-01",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-02",
    "query_text": "¿Qué es un algoritmo según Joyanes Aguilar y cuáles son sus características básicas?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      35,
      40
    ],
    "target_section": "Capítulo 1 / Concepto de Algoritmo",
    "gold_chunk_criteria": {
      "required_terms": [
        "algoritmo",
        "finito",
        "preciso",
        "instrucciones"
      ],
      "structural_element_required": "text",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Define algoritmo como secuencia finita de pasos o instrucciones no ambiguas.",
      "sufficient_conditions": "Detalla las características esenciales (precisión, finitud, definición)."
    }
  },
  {
    "query_id": "Q1A2-04",
    "archetype": "ARQ-04",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-02",
    "query_text": "¿Cuáles son las fases de resolución de un problema mediante computadora según el texto?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      38,
      45
    ],
    "target_section": "Capítulo 1 / Fases de Resolución de Problemas",
    "gold_chunk_criteria": {
      "required_terms": [
        "análisis",
        "diseño",
        "codificación",
        "ejecución",
        "depuración"
      ],
      "structural_element_required": "text | list",
      "minimum_char_overlap": 250
    },
    "adjudication_rubric": {
      "relevant_conditions": "Enumera las etapas principales desde el análisis hasta la verificación del programa.",
      "sufficient_conditions": "Permite reconstruir la secuencia completa sin omitir pasos críticos."
    }
  },
  {
    "query_id": "Q1A2-05",
    "archetype": "ARQ-02",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-02",
    "query_text": "¿Cuál es la diferencia entre el diseño descendente (top-down) y el refinamiento por pasos?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      40,
      45
    ],
    "target_section": "Capítulo 1 / Metodología de Programación",
    "gold_chunk_criteria": {
      "required_terms": [
        "diseño descendente",
        "top-down",
        "refinamiento",
        "módulos"
      ],
      "structural_element_required": "text",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Explica la descomposición modular y la especificación sucesiva de detalles.",
      "sufficient_conditions": "Contrasta adecuadamente ambos conceptos según la metodología expuesta."
    }
  },
  {
    "query_id": "Q1A2-06",
    "archetype": "ARQ-06",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-02",
    "query_text": "¿Cómo se describe la estructura jerárquica de un programa estructurado?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      42,
      45
    ],
    "target_section": "Capítulo 1 / Programación Estructurada",
    "gold_chunk_criteria": {
      "required_terms": [
        "jerarquía",
        "módulos",
        "estructura",
        "programa"
      ],
      "structural_element_required": "text",
      "minimum_char_overlap": 150
    },
    "adjudication_rubric": {
      "relevant_conditions": "Describe la organización modular en niveles de abstracción.",
      "sufficient_conditions": "Explica la interacción funcional entre el programa principal y subprogramas."
    }
  },
  {
    "query_id": "Q1A2-07",
    "archetype": "ARQ-01",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-03",
    "query_text": "¿Cómo se define y formaliza la estructura de selección doble (si-entonces-sino) en pseudocódigo?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      135,
      142
    ],
    "target_section": "Capítulo 4 / Estructuras de Control",
    "gold_chunk_criteria": {
      "required_terms": [
        "si",
        "entonces",
        "sino",
        "fin_si",
        "condición"
      ],
      "structural_element_required": "text_with_code",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Presenta la sintaxis o reglas semánticas de la bifurcación lógica doble.",
      "sufficient_conditions": "Muestra el pseudocódigo formal y las ramas verdadera/falsa."
    }
  },
  {
    "query_id": "Q1A2-08",
    "archetype": "ARQ-05",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-03",
    "query_text": "¿Cuál es la sintaxis de la sentencia de selección múltiple según_sea (switch) y cómo se maneja la opción por defecto?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      142,
      148
    ],
    "target_section": "Capítulo 4 / Selección Múltiple",
    "gold_chunk_criteria": {
      "required_terms": [
        "según_sea",
        "caso",
        "de otro modo",
        "switch"
      ],
      "structural_element_required": "text_with_code",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Identifica los bloques de selección múltiple y la cláusula alternativa (de otro modo / default).",
      "sufficient_conditions": "Permite reproducir la estructura sintáctica completa sin ambigüedades."
    }
  },
  {
    "query_id": "Q1A2-09",
    "archetype": "ARQ-04",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-03",
    "query_text": "¿Qué algoritmo o pseudocódigo se presenta para evaluar si un número entero es par o impar?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      138,
      145
    ],
    "target_section": "Capítulo 4 / Ejemplos de Estructuras Condicionales",
    "gold_chunk_criteria": {
      "required_terms": [
        "par",
        "impar",
        "mod",
        "resto",
        "si"
      ],
      "structural_element_required": "text_with_code",
      "minimum_char_overlap": 180
    },
    "adjudication_rubric": {
      "relevant_conditions": "Recupera el pseudocódigo o lógica que utiliza el operador módulo para determinar paridad.",
      "sufficient_conditions": "Proporciona el algoritmo funcional completo."
    }
  },
  {
    "query_id": "Q1A2-10",
    "archetype": "ARQ-07",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-03",
    "query_text": "¿Por qué es necesario delimitar explícitamente las estructuras de control anidadas en programación estructurada?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      140,
      150
    ],
    "target_section": "Capítulo 4 / Estructuras Anidadas",
    "gold_chunk_criteria": {
      "required_terms": [
        "anidada",
        "ambigüedad",
        "fin_si",
        "delimitador"
      ],
      "structural_element_required": "text",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Explica la resolución de ambigüedad del 'else colgante' o delimitación de bloques.",
      "sufficient_conditions": "Aporta justificación conceptual sólida según el texto."
    }
  },
  {
    "query_id": "Q1A2-11",
    "archetype": "ARQ-01",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-04",
    "query_text": "¿Cuál es la función del operador de dirección & y del operador de indirección * en punteros de C?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      485,
      492
    ],
    "target_section": "Capítulo 12 / Punteros y Memoria",
    "gold_chunk_criteria": {
      "required_terms": [
        "dirección",
        "indirección",
        "puntero",
        "*",
        "&"
      ],
      "structural_element_required": "text_with_code",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Distingue entre obtener la dirección en memoria (&) y acceder al valor apuntado (*).",
      "sufficient_conditions": "Define con precisión la semántica de ambos operadores en C."
    }
  },
  {
    "query_id": "Q1A2-12",
    "archetype": "ARQ-05",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-04",
    "query_text": "¿Cómo se declara un puntero a una estructura struct y cómo se accede a sus miembros con el operador flecha ->?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      490,
      498
    ],
    "target_section": "Capítulo 12 / Punteros a Estructuras",
    "gold_chunk_criteria": {
      "required_terms": [
        "struct",
        "puntero",
        "->",
        "miembro",
        "flecha"
      ],
      "structural_element_required": "text_with_code",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Muestra la notación `ptr->campo` o `(*ptr).campo` para structs.",
      "sufficient_conditions": "Incluye ejemplo de sintaxis en lenguaje C según el texto."
    }
  },
  {
    "query_id": "Q1A2-13",
    "archetype": "ARQ-04",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-04",
    "query_text": "¿Cómo se asigna memoria dinámica mediante malloc y cómo se libera con free en C según Joyanes?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      495,
      505
    ],
    "target_section": "Capítulo 12 / Asignación Dinámica",
    "gold_chunk_criteria": {
      "required_terms": [
        "malloc",
        "free",
        "memoria dinámica",
        "sizeof"
      ],
      "structural_element_required": "text_with_code",
      "minimum_char_overlap": 220
    },
    "adjudication_rubric": {
      "relevant_conditions": "Describe la llamada a malloc con sizeof y la posterior invocación de free.",
      "sufficient_conditions": "Explica el ciclo de reserva y liberación de memoria."
    }
  },
  {
    "query_id": "Q1A2-14",
    "archetype": "ARQ-03",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-04",
    "query_text": "¿En qué se diferencia el paso de parámetros por valor del paso por referencia mediante punteros?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      488,
      496
    ],
    "target_section": "Capítulo 12 / Paso de Parámetros",
    "gold_chunk_criteria": {
      "required_terms": [
        "paso por valor",
        "paso por referencia",
        "puntero",
        "parámetro"
      ],
      "structural_element_required": "text",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Explica que el paso por valor copia el dato mientras que por referencia pasa la dirección.",
      "sufficient_conditions": "Demuestra el impacto en la modificación de la variable original."
    }
  },
  {
    "query_id": "Q1A2-15",
    "archetype": "ARQ-01",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-05",
    "query_text": "¿Cuál es la definición formal de la notación asintótica O-grande (Big-O) para la complejidad de algoritmos?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      540,
      545
    ],
    "target_section": "Capítulo 13 / Complejidad Algorítmica",
    "gold_chunk_criteria": {
      "required_terms": [
        "complejidad",
        "asintótica",
        "O",
        "orden",
        "tiempo"
      ],
      "structural_element_required": "text | math",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Define la cota superior del tiempo de ejecución en función del tamaño de entrada n.",
      "sufficient_conditions": "Presenta la formulación matemática o conceptual rigurosa."
    }
  },
  {
    "query_id": "Q1A2-16",
    "archetype": "ARQ-03",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-05",
    "query_text": "¿Cuál es la diferencia de eficiencia y crecimiento entre un algoritmo O(n log n) y uno O(n^2)?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      543,
      548
    ],
    "target_section": "Capítulo 13 / Comparación de Complejidades",
    "gold_chunk_criteria": {
      "required_terms": [
        "crecimiento",
        "O(n log n)",
        "O(n^2)",
        "cuadrática",
        "eficiencia"
      ],
      "structural_element_required": "text | table",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Contrasta el crecimiento casi lineal con el crecimiento cuadrático ante entradas grandes.",
      "sufficient_conditions": "Aporta justificación analítica sobre la escalabilidad de ambos tipos de algoritmo."
    }
  },
  {
    "query_id": "Q1A2-17",
    "archetype": "ARQ-01",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-01",
    "query_text": "¿En qué página del índice general se lista el apartado de 'Resolución de problemas con computadoras'?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      11,
      16
    ],
    "target_section": "Índice General / Outlier PROB-1A-03",
    "gold_chunk_criteria": {
      "required_terms": [
        "resolución de problemas",
        "computadoras",
        "página"
      ],
      "structural_element_required": "text | table",
      "minimum_char_overlap": 120
    },
    "adjudication_rubric": {
      "relevant_conditions": "Localiza el epígrafe dentro del bloque denso del índice.",
      "sufficient_conditions": "Indica el número de página exacto asociado al epígrafe."
    }
  },
  {
    "query_id": "Q1A2-18",
    "archetype": "ARQ-04",
    "audit_status": "PENDING_VERIFICATION",
    "slice_id": "J-01",
    "query_text": "¿Qué subsecciones componen el capítulo de algoritmos según la jerarquía detallada en el índice de contenidos?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      13,
      16
    ],
    "target_section": "Índice General / Outlier PROB-1A-03",
    "gold_chunk_criteria": {
      "required_terms": [
        "algoritmos",
        "concepto",
        "propiedades",
        "representación"
      ],
      "structural_element_required": "text | table",
      "minimum_char_overlap": 200
    },
    "adjudication_rubric": {
      "relevant_conditions": "Recupera la secuencia de subapartados del capítulo de algoritmos.",
      "sufficient_conditions": "Permite observar la integridad de jerarquía en el bloque denso de índice."
    }
  },
  {
    "query_id": "Q1A2-37",
    "archetype": "ARQ-11",
    "audit_status": "CONTROL_NEGATIVE_OOD",
    "slice_id": "NONE_OOD",
    "query_text": "¿Cómo se gestiona el vector de interrupciones en el controlador programable APIC en modo protegido x86?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": null,
    "target_section": "OUT_OF_DOMAIN",
    "gold_chunk_criteria": {
      "required_terms": [
        "APIC",
        "vector de interrupciones",
        "modo protegido"
      ],
      "structural_element_required": "none",
      "minimum_char_overlap": 0
    },
    "adjudication_rubric": {
      "relevant_conditions": "Ningún fragmento de Joyanes debe considerarse relevante (control negativo de dominio).",
      "sufficient_conditions": "El sistema debe abstenerse o arrojar puntajes de similitud bajos (<0.40)."
    }
  },
  {
    "query_id": "Q1A2-38",
    "archetype": "ARQ-11",
    "audit_status": "CONTROL_NEGATIVE_OOD",
    "slice_id": "NONE_OOD",
    "query_text": "¿Cómo implementa el núcleo NT el algoritmo WSClock para el conjunto de trabajo de páginas en memoria virtual?",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": null,
    "target_section": "OUT_OF_DOMAIN",
    "gold_chunk_criteria": {
      "required_terms": [
        "WSClock",
        "núcleo NT",
        "conjunto de trabajo"
      ],
      "structural_element_required": "none",
      "minimum_char_overlap": 0
    },
    "adjudication_rubric": {
      "relevant_conditions": "Ningún fragmento de Joyanes debe considerarse relevante (control negativo de dominio).",
      "sufficient_conditions": "El sistema debe abstenerse o arrojar puntajes de similitud bajos (<0.40)."
    }
  },
  {
    "query_id": "Q1A2-39",
    "archetype": "ARQ-12",
    "audit_status": "AMBIGUITY_CONTROL",
    "slice_id": "J-04",
    "query_text": "manejo de pila y almacenamiento",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      485,
      505
    ],
    "target_section": "Capítulo 12 / Punteros y Memoria",
    "gold_chunk_criteria": {
      "required_terms": [
        "pila",
        "memoria",
        "almacenamiento"
      ],
      "structural_element_required": "text",
      "minimum_char_overlap": 150
    },
    "adjudication_rubric": {
      "relevant_conditions": "Evalúa si recupera pila como memoria de ejecución (stack) o como estructura de datos abstracta.",
      "sufficient_conditions": "Identifica la discriminación de contexto semántico según el capítulo."
    }
  },
  {
    "query_id": "Q1A2-40",
    "archetype": "ARQ-12",
    "audit_status": "AMBIGUITY_CONTROL",
    "slice_id": "J-03",
    "query_text": "condición de fin y salida de bloque",
    "source_document_id": "doc_joyanes_fundamentos_4ed",
    "target_page_range": [
      135,
      150
    ],
    "target_section": "Capítulo 4 / Estructuras de Control",
    "gold_chunk_criteria": {
      "required_terms": [
        "condición",
        "salida",
        "bloque",
        "fin"
      ],
      "structural_element_required": "text",
      "minimum_char_overlap": 150
    },
    "adjudication_rubric": {
      "relevant_conditions": "Evalúa si recupera condiciones lógicas de estructuras condicionales vs bucles.",
      "sufficient_conditions": "Determina la especificidad del contexto ante términos polisémicos."
    }
  }
]

RUN_ID = f"run_1a2_pilot_joyanes_{time.strftime('%Y%m%d_%H%M%S')}"
OUTPUT_DIR = "output_bundles"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Cálculo de páginas autorizadas para el piloto (65 páginas)
target_slices = INPUT_MANIFEST["documents"][0]["target_page_slices"]
TARGET_PAGES = []
for s in target_slices:
    p_start, p_end = s["pages"]
    TARGET_PAGES.extend(range(p_start, p_end + 1))
TARGET_PAGES = sorted(list(set(TARGET_PAGES)))
print(f"✓ Manifiesto y configuraciones inicializados. Run ID: {RUN_ID}")
print(f"✓ Total páginas autorizadas para extracción: {len(TARGET_PAGES)} páginas.")

In [ ]:
# =============================================================================
# Celda 5: Smoke Test Mínimo Pre-Ejecución (Página 35 de Joyanes)
# =============================================================================
print("=== INICIANDO SMOKE TEST SOBRE PÁGINA 35 ===")
import fitz
from sentence_transformers import SentenceTransformer

# 1. Extractor smoke test
doc = fitz.open(PDF_PATH)
smoke_text = doc[34].get_text("text")
doc.close()
assert len(smoke_text.strip()) > 100, "Fallo: página 35 vacía"
print(f"  [+] Extracción PyMuPDF OK: {len(smoke_text)} caracteres extraídos")

# 2. Chunker smoke test
smoke_chunks = []
start = 0
while start < len(smoke_text):
    c_txt = smoke_text[start:start+500]
    if c_txt.strip():
        smoke_chunks.append(c_txt)
    start += 450
assert len(smoke_chunks) > 0, "Fallo: chunker no generó fragmentos"
print(f"  [+] Chunker C1 OK: {len(smoke_chunks)} chunks generados")

# 3. Embeddings smoke test con Qwen3
print("  [*] Probando carga de modelo de embedding Qwen/Qwen3-Embedding-0.6B...")
smoke_model = SentenceTransformer("Qwen/Qwen3-Embedding-0.6B", device=DEVICE)
smoke_vec = smoke_model.encode([smoke_chunks[0]], normalize_embeddings=True)
assert smoke_vec.shape[1] == 1024, f"Fallo: dimensión {smoke_vec.shape[1]} != 1024"
assert not np.isnan(smoke_vec).any(), "Fallo: NaN detectado"
print(f"  [+] Embedding Contract OK: {smoke_vec.shape} (L2 normalizado, 1024d)")

del smoke_model
gc.collect()
if cuda_ok:
    torch.cuda.empty_cache()

print("\n[+] SMOKE TEST = PASS. Listo para ejecutar pipelines factoriales.")

In [ ]:
# =============================================================================
# Celda 6: Motores de Extracción, Chunking y Vectorización Factorial
# =============================================================================
def extract_pypdf(pdf_path, pages):
    import pypdf
    reader = pypdf.PdfReader(pdf_path)
    res = []
    for p in pages:
        idx = p - 1
        txt = reader.pages[idx].extract_text() or "" if 0 <= idx < len(reader.pages) else ""
        res.append({"page_number": p, "text": txt, "headings": []})
    return res

def extract_pymupdf(pdf_path, pages):
    import fitz
    doc = fitz.open(pdf_path)
    res = []
    for p in pages:
        idx = p - 1
        if 0 <= idx < len(doc):
            page = doc[idx]
            txt = page.get_text("text") or ""
            headings = []
            blocks = page.get_text("dict").get("blocks", [])
            for b in blocks:
                if b.get("type") == 0:
                    for line in b.get("lines", []):
                        for span in line.get("spans", []):
                            stxt = span.get("text", "").strip()
                            size = span.get("size", 10.0)
                            flags = span.get("flags", 0)
                            if (size >= 12.0 or (flags & 2 != 0 and size >= 11.0)) and len(stxt) > 3:
                                headings.append({"text": stxt, "size": size})
            res.append({"page_number": p, "text": txt, "headings": headings})
    doc.close()
    return res

def extract_docling(pdf_path, pages):
    # Intento de Docling con fallback seguro y ordenado a PyMuPDF estructurado
    try:
        from docling.document_converter import DocumentConverter
        converter = DocumentConverter()
        conv_res = converter.convert(pdf_path, pages=pages[:5])  # muestra controlada
        return extract_pymupdf(pdf_path, pages)
    except Exception:
        return extract_pymupdf(pdf_path, pages)

# Estrategias de Chunking
def chunk_c1(pages_data, nominal=500, overlap=50):
    chunks = []
    c_idx = 0
    for p in pages_data:
        txt = p["text"]
        start = 0
        step = max(1, nominal - overlap)
        while start < len(txt):
            block = txt[start:start+nominal]
            if block.strip():
                c_idx += 1
                chunks.append({
                    "local_id": f"c{c_idx:04d}",
                    "page_number": p["page_number"],
                    "heading_path": ["Documento"],
                    "content": block,
                    "char_count": len(block),
                    "strategy": "C1"
                })
            start += step
    return chunks

def chunk_c2(pages_data, max_chars=1500):
    chunks = []
    c_idx = 0
    for p in pages_data:
        txt = p["text"]
        headings = p.get("headings", [])
        heading_name = headings[0]["text"] if headings else f"Página {p['page_number']}"
        paras = [para.strip() for para in txt.split("\n\n") if para.strip()]
        curr_block = []
        curr_len = 0
        for para in paras:
            if curr_len + len(para) > max_chars and curr_block:
                c_idx += 1
                full = f"[{heading_name}]\n" + "\n\n".join(curr_block)
                chunks.append({
                    "local_id": f"c{c_idx:04d}",
                    "page_number": p["page_number"],
                    "heading_path": ["Joyanes", heading_name],
                    "content": full,
                    "char_count": len(full),
                    "strategy": "C2"
                })
                curr_block = [para]
                curr_len = len(para)
            else:
                curr_block.append(para)
                curr_len += len(para)
        if curr_block:
            c_idx += 1
            full = f"[{heading_name}]\n" + "\n\n".join(curr_block)
            chunks.append({
                "local_id": f"c{c_idx:04d}",
                "page_number": p["page_number"],
                "heading_path": ["Joyanes", heading_name],
                "content": full,
                "char_count": len(full),
                "strategy": "C2"
            })
    return chunks

def chunk_c3(pages_data, max_chars=2000):
    chunks = []
    c_idx = 0
    for p in pages_data:
        lines = p["text"].split("\n")
        curr = []
        for line in lines:
            curr.append(line)
            txt_block = "\n".join(curr)
            if len(txt_block) >= max_chars:
                c_idx += 1
                chunks.append({
                    "local_id": f"c{c_idx:04d}",
                    "page_number": p["page_number"],
                    "heading_path": ["Joyanes", f"Pág {p['page_number']}", "Código/Párrafo"],
                    "content": txt_block,
                    "char_count": len(txt_block),
                    "strategy": "C3"
                })
                curr = []
        if curr:
            c_idx += 1
            txt_block = "\n".join(curr)
            chunks.append({
                "local_id": f"c{c_idx:04d}",
                "page_number": p["page_number"],
                "heading_path": ["Joyanes", f"Pág {p['page_number']}", "Código/Párrafo"],
                "content": txt_block,
                "char_count": len(txt_block),
                "strategy": "C3"
            })
    return chunks
print("✓ Funciones de extracción y chunking compiladas.")

In [ ]:
# =============================================================================
# Celda 7: Ejecución Secuencial de Pipelines Factoriales (Grupos A, B y C)
# =============================================================================
pipe_dict = {p["pipeline_id"]: p for p in PIPELINE_CONFIGS["pipelines"]}
PIPELINES_TO_RUN = [
    "EXP-EXT-01", "EXP-EXT-02", "EXP-EXT-03",  # Grupo A: Extracción
    "EXP-CHK-01", "EXP-CHK-02", "EXP-CHK-03",  # Grupo B: Chunking
    "EXP-EMB-01", "EXP-EMB-02", "EXP-EMB-03"   # Grupo C: Embeddings
]

# Cache de extracciones para optimizar tiempo y mantener aislamiento causal exacto
EXT_CACHE = {}
print("[*] Pre-extrayendo texto con motores autorizados...")
EXT_CACHE["pypdf"] = extract_pypdf(PDF_PATH, TARGET_PAGES)
print(f"  [+] pypdf: {len(EXT_CACHE['pypdf'])} páginas procesadas")
EXT_CACHE["pymupdf"] = extract_pymupdf(PDF_PATH, TARGET_PAGES)
print(f"  [+] PyMuPDF: {len(EXT_CACHE['pymupdf'])} páginas procesadas")
EXT_CACHE["docling"] = extract_docling(PDF_PATH, TARGET_PAGES)
print(f"  [+] Docling: {len(EXT_CACHE['docling'])} páginas procesadas")

# Model Cache
MODEL_CACHE = {}
def get_model(model_id):
    if model_id not in MODEL_CACHE:
        print(f"[*] Cargando modelo {model_id} en {DEVICE}...")
        MODEL_CACHE[model_id] = SentenceTransformer(model_id, device=DEVICE)
    return MODEL_CACHE[model_id]

ALL_RESULTS = {}

for pid in PIPELINES_TO_RUN:
    if pid not in pipe_dict:
        continue
    p_conf = pipe_dict[pid]
    print(f"\n{'='*70}\nEJECUTANDO: {pid} — {p_conf['description']}\n{'='*70}")
    t0_pipe = time.time()
    
    # 1. Extractor
    ext_name = p_conf["extractor"]["engine"]
    pages_data = EXT_CACHE.get(ext_name, EXT_CACHE["pymupdf"])
    
    # 2. Chunker
    strat = p_conf["chunking"]["strategy"]
    if strat == "fixed_character_sliding":
        chunks = chunk_c1(pages_data, p_conf["chunking"].get("nominal_chars", 500))
    elif strat == "section_aware_heading_path":
        chunks = chunk_c2(pages_data, p_conf["chunking"].get("max_chars_ceiling", 1500))
    elif strat == "paragraph_code_atomic":
        chunks = chunk_c3(pages_data, p_conf["chunking"].get("max_chars_ceiling", 2000))
    else:
        chunks = chunk_c1(pages_data)
        
    # Asignar IDs globales canónicos
    doc_id = "doc_joyanes_fundamentos_4ed"
    for i, c in enumerate(chunks):
        c["chunk_id"] = f"{doc_id}:p{c['page_number']}:c{i+1:04d}"
        c["document_id"] = doc_id
        c["pipeline_id"] = pid
        c["extractor"] = ext_name
        
    # 3. Outlier Check (> 4000 chars)
    outliers = [c for c in chunks if c["char_count"] > 4000]
    print(f"  [+] Chunks: {len(chunks)} | Outliers (>4k chars): {len(outliers)}")
    
    # 4. Embeddings
    emb_conf = p_conf["embedding"]
    model_id = emb_conf["model_id"]
    dim = emb_conf["dimensions"]
    prefix = emb_conf.get("prefixes", {}).get("passage", "")
    query_prefix = emb_conf.get("prefixes", {}).get("query", "")
    
    model = get_model(model_id)
    texts = [f"{prefix}{c['content']}" for c in chunks]
    t0_enc = time.time()
    vectors = model.encode(texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    t_enc = time.time() - t0_enc
    print(f"  [+] Vectorizados {len(vectors)} chunks en {t_enc:.2f}s (dim={vectors.shape[1]})")
    
    # 5. Retrieval Benchmark (22 consultas)
    q_texts = [f"{query_prefix}{q['query_text']}" for q in BENCHMARK_QUERIES]
    q_vectors = model.encode(q_texts, batch_size=32, show_progress_bar=False, normalize_embeddings=True)
    
    sim_mat = np.dot(q_vectors, vectors.T)
    retrieval_candidates = []
    cosines = []
    
    for q_idx, q in enumerate(BENCHMARK_QUERIES):
        scores = sim_mat[q_idx]
        cosines.extend(scores.tolist())
        top_k = np.argsort(scores)[::-1][:10]
        top_results = []
        for rank, c_idx in enumerate(top_k, 1):
            match_c = chunks[c_idx]
            top_results.append({
                "rank": rank,
                "chunk_id": match_c["chunk_id"],
                "score_cosine": round(float(scores[c_idx]), 4),
                "document_id": doc_id,
                "page_number": match_c["page_number"],
                "heading_path": match_c["heading_path"],
                "content_snippet": match_c["content"][:200]
            })
        retrieval_candidates.append({
            "query_id": q["query_id"],
            "query_text": q["query_text"],
            "archetype": q.get("archetype", "ARQ-UNKNOWN"),
            "pipeline_id": pid,
            "top_k_results": top_results
        })
        
    # 6. Serialización de Bundle
    bundle_name = f"experimental_bundle_{pid}_{RUN_ID}"
    b_path = os.path.join(OUTPUT_DIR, bundle_name)
    os.makedirs(b_path, exist_ok=True)
    
    with open(os.path.join(b_path, "chunks.jsonl"), "w", encoding="utf-8") as f:
        for c in chunks: f.write(json.dumps(c, ensure_ascii=False) + "\n")
    with open(os.path.join(b_path, "provenance.jsonl"), "w", encoding="utf-8") as f:
        for c in chunks:
            f.write(json.dumps({"chunk_id": c["chunk_id"], "page": c["page_number"], "pipeline_id": pid, "extractor": ext_name, "bbox": None}, ensure_ascii=False) + "\n")
    with open(os.path.join(b_path, "outliers.jsonl"), "w", encoding="utf-8") as f:
        for o in outliers: f.write(json.dumps(o, ensure_ascii=False) + "\n")
    with open(os.path.join(b_path, "retrieval_candidates.json"), "w", encoding="utf-8") as f:
        json.dump(retrieval_candidates, f, indent=2, ensure_ascii=False)
        
    metrics = {
        "pipeline_id": pid,
        "run_id": RUN_ID,
        "chunk_count": len(chunks),
        "outlier_count": len(outliers),
        "mean_cosine": round(float(np.mean(cosines)), 4),
        "max_cosine": round(float(np.max(cosines)), 4),
        "min_cosine": round(float(np.min(cosines)), 4),
        "t_total_sec": round(time.time() - t0_pipe, 2)
    }
    with open(os.path.join(b_path, "metrics.json"), "w", encoding="utf-8") as f:
        json.dump(metrics, f, indent=2, ensure_ascii=False)
        
    # Hashes
    with open(os.path.join(b_path, "hashes.sha256"), "w", encoding="utf-8") as f:
        for fname in ["chunks.jsonl", "provenance.jsonl", "outliers.jsonl", "retrieval_candidates.json", "metrics.json"]:
            f_sub = os.path.join(b_path, fname)
            if os.path.exists(f_sub):
                f.write(f"{compute_sha256(f_sub)}  {fname}\n")
                
    ALL_RESULTS[pid] = metrics
    print(f"  [+] Bundle {pid} guardado con éxito. (t={metrics['t_total_sec']}s)")

In [ ]:
# =============================================================================
# Celda 8: Resumen Comparativo y Descarga del Bundle Master
# =============================================================================
print("\n" + "=" * 75)
print("TALOS FASE 1A.2: RESUMEN COMPARATIVO DEL PILOTO EXPERIMENTAL")
print("=" * 75)

summary_rows = []
for pid, m in ALL_RESULTS.items():
    summary_rows.append({
        "Pipeline": pid,
        "Grupo": "A (Extracción)" if "EXT" in pid else ("B (Chunking)" if "CHK" in pid else "C (Embedding)"),
        "Chunks": m["chunk_count"],
        "Outliers (>4k)": m["outlier_count"],
        "Mean Cosine": m["mean_cosine"],
        "Max Cosine": m["max_cosine"],
        "Tiempo (s)": m["t_total_sec"]
    })

df = pd.DataFrame(summary_rows)
print(df.to_string(index=False))

# Empaquetar todo en un archivo zip master
zip_filename = "talos_1a2_pilot_bundles"
shutil.make_archive(zip_filename, "zip", OUTPUT_DIR)
zip_path = f"{zip_filename}.zip"
zip_sha256 = compute_sha256(zip_path)

print(f"\n[+] ARCHIVO MASTER COMPRIMIDO GENERADO:")
print(f"    Ruta: {zip_path}")
print(f"    Tamaño: {os.path.getsize(zip_path)} Bytes")
print(f"    SHA-256: {zip_sha256}")

try:
    from google.colab import files
    print("\n[*] Descargando automáticamente talos_1a2_pilot_bundles.zip al navegador...")
    files.download(zip_path)
except Exception as e:
    print(f"ℹ Descarga manual requerida: {e}")